In [17]:
import os
import sys
sys.path.append('/Users/mariana/Documents/projects/Huawei/survan')
# sys.path.append('/home/mvargas/code/Huawei/survan')

from deep_lambda_cox import DeepLambdaSA
from lambda_cox import LambdaSA
from baseline_cox import SA
from utils import concordance_index

import yaml
import jax
import jax.numpy as jnp
import numpy as np
import pickle

In [9]:
root_pth = '/Users/mariana/Documents/projects/Huawei/Results/with_landmark_1/'
config_path = os.path.join(root_pth, 'config.yaml')
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['dataset_kwargs']['data_path'] = '/Users/mariana/Documents/projects/Huawei/SurvanData/mixed_tasks/mixed/H_100/dataset.h5'

In [23]:
model_path = os.path.join(root_pth, 'mixed_tasks/seed_1/model.pt')

In [24]:
model = DeepLambdaSA(config, seed=1)

In [25]:
params = pickle.load(open(model_path, 'rb'))

In [28]:
model.state = model.state.replace(params=params)

In [30]:
train_gen, test_gen = model.get_train_test()
seqs = test_gen.X
ts = test_gen.ts
cs = test_gen.cs

In [31]:
surv = model.survival_curve(seqs)
bs = model.integrated_brier_score(surv[:,0], ts, cs)
beta = model.state.params['cox_linear_model']['beta']
scores = model.scores(jnp.expand_dims(seqs[:,0], axis=1)).squeeze()
ci = concordance_index(scores, ts, cs)

In [32]:
bs

Array(0.04968122, dtype=float32)

In [33]:
ci

0.9252754275391392